In [1]:
from transformers import AutoModel, AutoTokenizer
import torch

model = AutoModel.from_pretrained("facebook/mms-tts-tam")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-tam")

In [2]:
import os
import pandas as pd
import torchaudio
import librosa
import numpy as np
from jiwer import wer, cer

metadata = [
    ("ID", "Text"),
    ("train_tamilmale_00730", "அவனை அவசியம் கண்டுபிடிக்க வேண்டும் சேனாபதி மௌனமாயிருந்தார்."),
    ("train_tamilmale_00731", "பிறகு மாமல்லர் விசாலமான அந்த அழகிய அரண்மனைக்குள் பிரவேசித்து தீபம் ஏந்துவோர் தம்பைப் பின் தொடர்ந்து வருவதற்குத் திணறும்படியாக அவ்வளவு விரைவாக நடந்து சென்றனர்."),
    ("train_tamilmale_00732", "பல்லவ சாம்ராஜ்யத்தின் பட்டத்தரசியான புவன மகாதேவி அந்தப்புர வாசற்படியில் வந்து நின்றார்."),
    ("train_tamilmale_00733", "அன்று ஏதோ சொல்லத் தொடங்குவதற்குள் மாமல்லர் அம்மா உங்களை ரொம்பவும் கெஞ்சிக் கேட்டுக் கொள்கிறேன் ஒரு வரம் தர வேண்டும் என்றார்."),
    ("train_tamilmale_00734", "அதற்குப் பிரதியாக நானும் ஒரு வரம் கேட்பேன் அதை நீ தரவேண்டும் என்று அன்பு கனிந்த குரலில் கூறினார்."),
    ("train_tamilmale_00735", "எனவே பகல் போஜன நேரத்தில் அரண்மனைவாசிகள் ஒருவரோடொருவர் அளவளாவுதல் இயலாத காரியம்."),
    ("train_tamilmale_00736", "நமது எல்லைக் காவல் படைகளைப் புலிகேசியின் ராட்சத சைனியம் வெகு சீக்கிரத்தில் முறியடித்து விட்டு அதிவேகமாக முன்னேறி வருகிறதாம்."),
    ("train_tamilmale_00737", "பல்லவ ராஜ்யத்துக்கு வந்திருக்கும் அபாயம் மிகப் பெரியது."),
    ("train_tamilmale_00738", "அந்தக் கட்டிடந்தான் காஞ்சி நகருக்குள்ளிருந்த புத்த விஹாரங்களுக்குள் மிகப் பெரியது."),
    ("train_tamilmale_00739", "கருணாமூர்த்தியான புத்த பகவானின் திருப் பற்களில் ஒன்று அந்தக் கோயிலின் கர்ப்பக் கிருஹத்தில் பிரதிஷ்டை செய்யப்பட்டிருந்தது."),
    ("train_tamilmale_00740", "பிள்ளை பிழைக்கவே ஏராளாமான பொருட்செலவு செய்து விஹாரத்தைப் புதுப்பித்தான்."),
    ("train_tamilmale_00741", "அதே சமயத்தில் இராஜ விஹாரத்துக்கு எதிர் வரிசையிலிருந்த கட்டிடங்கள் இருண்ட நிழலிலிருந்து இரண்ட வெண்புரவிகள் வெளிப்பட்டு வந்தன."),
    ("train_tamilmale_00742", "இனிமேல் நள்ளிரவில் கிளம்ப வேண்டாம் சுவாமி சிஷ்யப் பிள்ளையிடமும் சொல்லி வையுங்கள்."),
    ("train_tamilmale_00743", "காஞ்சிக்கு நான் புதிதாயிற்றே பிக்ஷு பரஞ்சோதியின் காதோடு மகேந்திர சக்கரவர்த்தியும் அவருடைய மகன் மாமல்ல நரசிம்மனுந்தான் என்றார்."),
    ("train_tamilmale_00744", "ஏதேதோ பயங்கர துர்க்கனவுகள் தோன்றி தூக்கத்தைக் கெடுத்தன."),
    ("train_tamilmale_00745", "இவன் மகாவீரன் ஆவான் அல்லது மகாத்மா ஆவான் என்று யாரோ ஒருவர் சொன்னது போலிருந்தது."),
    ("train_tamilmale_00746", "கண்கட்டுச் சோதனை முடிந்தது என்று சொல்லிக் கொண்டே அடிகள் கட்டை அவிழ்த்தார்."),
    ("train_tamilmale_00747", "அந்த அகழியினால் கோட்டைப் பாதுகாப்புக்குத்தான் என்ன பிரயோஜனம் எதிரிகள் வந்தால் சுலபமாய் நீந்திவிடமாட்டார்களா."),
    ("train_tamilmale_00748", "படகு அகழியின் அக்கரையை அடைந்தது."),
    ("train_tamilmale_00749", "சிவனடியார்களும் வைஷ்ணவப் பெரியார்களும் ஸ்தல யாத்திரை என்ற வியாஜத்தில் தேசமெங்கும் பிரயாணம் செய்து சைவ வைஷ்ணவ சமயங்களைப் பரப்பி வந்தார்கள்."),
    ("train_tamilmale_00750", "இது காரணமாகத் தமிழகத்தில் சென்று சில நூற்றாண்டுகாலமாய் வேரூன்றியிருந்த புத்த சமண சமயங்களுக்கும் சைவ வைஷ்ணவ சமயங்களுக்கும் தீவிரப் போட்டி ஏற்பட்டது."),
    ("train_tamilmale_00751", "காஞ்சி மாநகரில் பிறந்து வளர்ந்த கலை பயின்ற ஆயனர் இளம் பிராயத்திலேயே மகா சிற்பி என்ற பெயர் பெற்றுவிட்டார்."),
    ("train_tamilmale_00752", "நகரத்தை விட்டு எங்கேயாவது ஏகாந்தமான பிரதேசத்துக்குப் போனாலொழிய மேற்படி மனோரதம் நிறைவேறுவது சாத்தியமாகாது என்பதையும் அவர் நன்கு உணர்ந்தார்."),
    ("train_tamilmale_00753", "சக்கரவர்த்தியும் மாமல்லரும் ஒருநாள் கடல்மலைத் துறைமுகத்துக்குச் சென்றிருந்தபோது கடற்கரையோரமாகப் பரந்து கிடந்த குன்றுகளும் பாறைகளும் அவர்களுடைய கவனத்தைக் கவர்ந்தன."),
    ("train_tamilmale_00754", "அரண்ய மத்தியில் அமைந்த ஆயனர் வீட்டின் உட்புறம் கண்கொள்ளாகக் காட்சியளித்தது."),
    ("train_tamilmale_00755", "அந்தச் சித்திரங்களில் ஸ்ரீநடராஜ மூர்த்தியின் நாதாந்த நடனம் தாண்டவ நடனம் குஞ்சித நடனம் ஊர்த்வ நடனம் ஆகிய தோற்றங்கள் அதிகமாக இருந்தன."),
    ("train_tamilmale_00756", "சித்திரங்களிலும் சிலைகளிலும் தோற்றமளித்த இளம் பெண்ணானவள் அங்கே சுயமாகவே தோன்றி கால் சதங்கை களீர் களீர் என்று சப்திக்க நடனமாடிக் கொண்டிருந்தாள் அவளுக்கெதிரே சற்றுத் தூரத்தில் ஆயனச் சிற்பியார் உட்கார்ந்து கண்கொட்டாத ஆர்வத்துடன் பார்த்துக்கொண்டிருந்தார்."),
    ("train_tamilmale_00757", "யார் ருத்ராச்சாரியார் சக்கரவர்த்திக்குப் பக்கத்தில் பெரிய வெள்ளைத் தாடியோடு உட்கார்ந்திருந்தாரே அவரா ஆம் அவர்தான் நம் மகேந்திர சக்கரவர்த்தியின் சங்கீத ஆசிரியர்."),
    ("train_tamilmale_00758", "இன்னும் அறுபது சிலைகள் அமைந்தவுடனே உனக்குத் தக்க மணாளனைத் தேடிக் கல்யாணம் செய்து கொடுத்துவிட்டு மறுகாரியம் பார்ப்பேன்."),
    ("train_tamilmale_00759", "அவள் இவ்விதம் சொல்லி வாய்மூடிய அதே சமயத்தில் வாசற்புறத்தில் புத்தம் சரணம் கச்சாமி என்ற குரல் கேட்டது."),
    ("train_tamilmale_00760", "ஆம் சுவாமி அப்புறம் பன்னிரண்டு ஹஸ்த வகைகளை அமைத்திருக்கிறேன்."),
    ("train_tamilmale_00761", "நடனத்துக்குரிய ஆடை ஆபரணங்களை அணிந்து நின்ற சிவகாமியின் நவ யௌவன சௌந்தர்யத்தின் ஒளி பரஞ்சோதியின் கண்களைக் கூசச் செய்தது."),
    ("train_tamilmale_00762", "சிவகாமி பரஞ்சோதியைப் பார்த்தவண்ணம் இவருக்கு நான் நன்றி செலுத்தப் போவதில்லை."),
    ("train_tamilmale_00763", "சிவகாமியின் இந்தக் கடுஞ்சொல் கேட்டுக்கொண்டிருந்த மூன்று பேரையும் சிறிது திடுக்கிடச் செய்தது."),
    ("train_tamilmale_00764", "முற்றத்துக் கூரைமேல் உட்கார்ந்திருந்த மயில் ஜிவ்வென்று பறந்து தரைக்கு வந்தது."),
    ("train_tamilmale_00765", "அது முடிகிறவரையில் நான் தாமரைக் குளத்துக்குப் போய்வருகிறேன் என்று சொல்லி விட்டு சிவகாமி சமையற்கட்டை தாண்டி சென்ற போது வீட்டின் கொல்லைப்புறத்தை அடைந்தாள்."),
    ("train_tamilmale_00766", "அந்த அதிசயம் இன்னதென்பது மின்வெட்டைப் போல் அவள் உள்ளத்தில் உதித்தது."),
    ("train_tamilmale_00767", " சிவகாமி புத்த பிக்ஷுவை நமஸ்கரித்தபோது அவர் ஆர்வம் ததும்பிய விழிகளால் அவளை விழுங்குபவர்போல் பார்த்துவிட்டு புத்த தேவர் அருளால் உன் கோரிக்கை நிறைவேறட்டும். " ),
    ("train_tamilmale_00768", " தங்களுக்குச் சாவகாசம் தானே இன்று பிக்ஷை இங்கேயே வைத்துக் கொள்ள வேண்டும் என்று கூறி ஆயனர் பிக்ஷுவின் முகத்தை ஏறிட்டுப் பார்த்தார். " ),
    ("train_tamilmale_00769", " அவள் அப்பா புத்த பகவான் அன்பு மதத்தையும் அஹிம்சா தர்மத்தையும் உபதேசித்தது உண்மைதான். " )
]


metadata = pd.DataFrame(metadata[1:], columns=metadata[0])

# Display the DataFrame
print(metadata)

                       ID                                               Text
0   train_tamilmale_00730  அவனை அவசியம் கண்டுபிடிக்க வேண்டும் சேனாபதி மௌன...
1   train_tamilmale_00731  பிறகு மாமல்லர் விசாலமான அந்த அழகிய அரண்மனைக்கு...
2   train_tamilmale_00732  பல்லவ சாம்ராஜ்யத்தின் பட்டத்தரசியான புவன மகாதே...
3   train_tamilmale_00733  அன்று ஏதோ சொல்லத் தொடங்குவதற்குள் மாமல்லர் அம்...
4   train_tamilmale_00734  அதற்குப் பிரதியாக நானும் ஒரு வரம் கேட்பேன் அதை...
5   train_tamilmale_00735  எனவே பகல் போஜன நேரத்தில் அரண்மனைவாசிகள் ஒருவரோ...
6   train_tamilmale_00736  நமது எல்லைக் காவல் படைகளைப் புலிகேசியின் ராட்ச...
7   train_tamilmale_00737  பல்லவ ராஜ்யத்துக்கு வந்திருக்கும் அபாயம் மிகப்...
8   train_tamilmale_00738  அந்தக் கட்டிடந்தான் காஞ்சி நகருக்குள்ளிருந்த ப...
9   train_tamilmale_00739  கருணாமூர்த்தியான புத்த பகவானின் திருப் பற்களில...
10  train_tamilmale_00740  பிள்ளை பிழைக்கவே ஏராளாமான பொருட்செலவு செய்து வ...
11  train_tamilmale_00741  அதே சமயத்தில் இராஜ விஹாரத்துக்கு எதிர் வரிசையி...

In [3]:
output_folder = "generated_wavs_tamil"
os.makedirs(output_folder, exist_ok=True)

In [4]:
import scipy
# Iterate through each row of the metadata to generate and save audio
for index, row in metadata.iterrows():
    text = row['Text']
    wav_name = f"{row['ID']}.wav"  # Add the .wav extension
    output_path = os.path.join(output_folder, wav_name)

    # Tokenize the input text
    inputs = tokenizer(text, return_tensors="pt")

    # Generate the audio waveform
    with torch.no_grad():
        output = model(**inputs).waveform

    # Save the generated audio as a .wav file
    scipy.io.wavfile.write(output_path, rate=model.config.sampling_rate, data=output.squeeze().numpy())

    print(f"Generated and saved: {output_path}")

Generated and saved: generated_wavs_tamil\train_tamilmale_00730.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00731.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00732.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00733.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00734.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00735.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00736.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00737.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00738.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00739.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00740.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00741.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00742.wav
Generated and saved: generated_wavs_tamil\train_tamilmale_00743.wav
Generated and saved: generated_wavs_tamil\train_

In [5]:
import os
import csv
import librosa
import numpy as np
from jiwer import cer, wer
import parselmouth

def compute_metrics(original_folder, generated_folder, output_csv):
    metrics = {
        "File": [], "MCD": [], "LSD": [], "SNR": [],
        "Pitch RMSE": [], "Duration Difference": [], "CER": [], "WER": []
    }
    
    original_files = sorted(os.listdir(original_folder))
    generated_files = sorted(os.listdir(generated_folder))
    
    for orig_file, gen_file in zip(original_files, generated_files):
        orig_path = os.path.join(original_folder, orig_file)
        gen_path = os.path.join(generated_folder, gen_file)
        
        # Load audio files
        orig_audio, orig_sr = librosa.load(orig_path, sr=None)
        gen_audio, gen_sr = librosa.load(gen_path, sr=None)
        duration_diff = abs(len(orig_audio) / orig_sr - len(gen_audio) / gen_sr)
        
        # Resample if needed
        if orig_sr != gen_sr:
            gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)
            gen_sr = orig_sr
        
        # Align audio lengths
        min_length = min(len(orig_audio), len(gen_audio))
        orig_audio = orig_audio[:min_length]
        gen_audio = gen_audio[:min_length]
        
        # Compute spectrograms
        orig_mel = librosa.feature.melspectrogram(y=orig_audio, sr=orig_sr)
        gen_mel = librosa.feature.melspectrogram(y=gen_audio, sr=gen_sr)
        
        # Align spectrogram shapes
        min_frames = min(orig_mel.shape[1], gen_mel.shape[1])
        orig_mel = orig_mel[:, :min_frames]
        gen_mel = gen_mel[:, :min_frames]
        
        # Compute metrics
        mcd = np.mean(np.abs(orig_mel - gen_mel))  # Simplified
        lsd = np.mean(np.abs(librosa.amplitude_to_db(orig_mel) - librosa.amplitude_to_db(gen_mel)))
        noise = orig_audio - gen_audio
        snr = 10 * np.log10(np.sum(orig_audio ** 2) / np.sum(noise ** 2))
        orig_pitch = parselmouth.Sound(orig_path).to_pitch().selected_array["frequency"]
        gen_pitch = parselmouth.Sound(gen_path).to_pitch().selected_array["frequency"]
        min_pitch_length = min(len(orig_pitch), len(gen_pitch))
        pitch_rmse = np.sqrt(np.mean((orig_pitch[:min_pitch_length] - gen_pitch[:min_pitch_length]) ** 2))
        # duration_diff = abs(len(orig_audio) / orig_sr - len(gen_audio) / gen_sr)
        orig_text = os.path.splitext(orig_file)[0]  # Assumes filename contains transcription
        gen_text = os.path.splitext(gen_file)[0]  # Assumes filename contains transcription
        char_error_rate = cer(orig_text, gen_text)
        word_error_rate = wer(orig_text, gen_text)
        
        # Append to metrics
        metrics["File"].append(orig_file)
        metrics["MCD"].append(mcd)
        metrics["LSD"].append(lsd)
        metrics["SNR"].append(snr)
        metrics["Pitch RMSE"].append(pitch_rmse)
        metrics["Duration Difference"].append(duration_diff)
        metrics["CER"].append(char_error_rate)
        metrics["WER"].append(word_error_rate)
    
    # Write metrics to a CSV file
    with open(output_csv, mode='w', newline='') as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(metrics.keys())  # Write header
        writer.writerows(zip(*metrics.values()))  # Write rows
    
    print(f"Metrics saved to {output_csv}")



# Folders containing original and generated wav files
original_folder = "D:/Wav2Lip-master/TTS Evaluation/wav_tamil"
generated_folder = "D:/Wav2Lip-master/TTS Evaluation/generated_wavs_tamil"

# Output CSV file
output_csv = "tts_model_evaluation_tamil.csv"

# Compute and save metrics
compute_metrics(original_folder, generated_folder, output_csv)


C:\Users\satvi\AppData\Local\Temp\ipykernel_10188\1648882432.py:28: FutureWarning: Pass orig_sr=16000, target_sr=48000 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)
C:\Users\satvi\AppData\Local\Temp\ipykernel_10188\1648882432.py:28: FutureWarning: Pass orig_sr=16000, target_sr=48000 as keyword args. From version 0.10 passing these as positional arguments will result in an error
  gen_audio = librosa.resample(gen_audio, gen_sr, orig_sr)


Metrics saved to tts_model_evaluation_tamil.csv
